In [1]:
import pandas as pd
import json
import datasets
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to the file TO CHANGE
path = '/projects/iris/rferreir/datasets/GeoSQA/final_dataset.csv'

# Path to the file dataset_release_no_image.json
original_data_path = '/projects/iris/rferreir/datasets/GeoSQA/original_data/dataset_release_no_image.json'

# Path to the Hugging Face dataset to save
output_path = '...'

In [3]:
df = pd.read_csv(path)
df.head()

,question_id,scenario_id,answer,categories,text,translated_text,annotation,scenario,question,answers,A,B,C,D
0,164,76,D,景观图,Image Annotation :\n2012年6月18日，载着景海鹏、刘旺、刘洋三名航天...,"Image Annotation:\nOn June 18, 2012, the Shenz...","On June 18, 2012, the Shenzhou-9 spacecraft ca...","CNTV News: On June 18, 2012, the Shenzhou-9 sp...",While the Tiangong-1 and Shenzhou-9 combinatio...,A) Extragalactic galaxy\nB) Milky Way\nC) Sola...,Extragalactic galaxy,Milky Way,Solar system,Earth-Moon system
1,165,76,A,景观图,Image Annotation :\n2012年6月18日，载着景海鹏、刘旺、刘洋三名航天...,"Image Annotation:\nOn June 18, 2012, the Shenz...","On June 18, 2012, the Shenzhou-9 spacecraft ca...","CNTV News: On June 18, 2012, the Shenzhou-9 sp...",When Tiangong-1 and Shenzhou-9 successfully do...,A) Harbin\nB) Beijing\nC) Guangzhou\nD) Shanghai,Harbin,Beijing,Guangzhou,Shanghai
2,166,76,B,景观图,Image Annotation :\n2012年6月18日，载着景海鹏、刘旺、刘洋三名航天...,"Image Annotation:\nOn June 18, 2012, the Shenz...","On June 18, 2012, the Shenzhou-9 spacecraft ca...","CNTV News: On June 18, 2012, the Shenzhou-9 sp...",Which of the following statements about the Ti...,A) The shadow of the Jiuquan satellite launch ...,The shadow of the Jiuquan satellite launch tow...,The Earth's orbital speed is increasing.,The flag-raising ceremony in Tiananmen Square ...,The Argentine Pampas are turning a withered ye...
3,646,309,D,地图,Image Annotation :\n主要商业区在城市的中部\nX在城区的西北\nY在城区...,Image Annotation:\nThe main commercial area is...,The main commercial area is in the center of t...,The figure below is a schematic diagram of the...,"With urban development, residential area Y is ...",A) ①②\nB) ③④\nC) ①③\nD) ②④,①②,③④,①③,②④
4,1092,522,C,等温线|等值线图|地图,Image Annotation :\n新疆南部附近等值线值为40度\n新疆北部附近等值线值...,Image Annotation:\nThe contour line value near...,The contour line value near southern Xinjiang ...,The following map shows the high temperature f...,"Based on the information in the map, which of ...","A) At this time, the high temperature range in...","At this time, the high temperature range in so...",Lhasa did not experience high temperatures due...,Changchun was affected by a strong cold front ...,There is a high probability of drought in Hang...


In [4]:
num_null_rows = df.isnull().any(axis=1).sum()

print(num_null_rows)

42


## Dealing with missing questions

In [19]:
num_null_rows = df['question'].isnull().sum()

print(f"Number of questions missing : {num_null_rows}")

null_rows = df[df["question"].isnull()]

zh_text = list(null_rows['text'])
en_text = list(null_rows['translated_text'])
ids = list(null_rows['question_id'])
for zh_t, en_t, idx in zip(zh_text, en_text, ids):
    print("\n---------------\n")
    print(idx)
    print(zh_t)
    print(en_t)

3

---------------

195
Image Annotation :
该地岩层中间新，两侧老。
河流所在岩层年龄为110万年。
①的表层年龄2100万年。
②处岩层断层。
③处河流流向是从下到上。
聚落在河流的西侧。


地处北纬50°附近欧洲中部的某聚落局部地区示意图

Scenario :
下图为地处北纬50°附近欧洲中部的某聚落局部地区示意图。读图，回答下列各题。

Question :
图中

Answers :
A) 河流形成于距今2300万年前
B) ①处经历了先侵蚀后沉积过程
C) ②处经历了水平挤压、断裂下陷等内力作用 
D) 河流③处左岸堆积，右岸侵蚀
Image Annotation:
The rock formations in this area are new in the middle and old on the sides.
The rock formations in which the river is located are 1.1 million years old.
The surface layer at ① is 21 million years old.
The rock formation at ② is faulted.
The river flows from bottom to top at ③.
The settlement is located on the west side of the river.

Schematic diagram of a settlement located near 50°N in central Europe

Scenario:
The following diagram shows a partial diagram of a settlement located near 50°N in central Europe. Read the diagram and answer the following questions.

Question:

Answers:
A) The river was formed 23 million years ago.
B) ① experienced erosion followed by sedimentati

In [20]:
t = """Image Annotation: The rock strata in this area are younger in the middle and older on both sides.
The rock strata containing the river are 1.1 million years old.
The surface layer at ① is 21 million years old.
There is a fault in the rock strata at ②.
The river flows from bottom to top at ③.
The settlement is on the west side of the river.
Schematic diagram of a partial area of a settlement in Central Europe, near 50°N latitude.

Scenario: The following diagram is a partial area of a settlement in Central Europe, near 50°N latitude. Refer to the diagram and answer the following questions.

Question: In the diagram

Answers:

A) The river formed 23 million years ago.
B) Area ① underwent erosion followed by deposition.
C) Area ② experienced horizontal compression, fault subsidence, and other internal forces.
D) At river ③, deposition occurred on the left bank, and erosion occurred on the right bank."""
df.loc[df['question_id']==195, 'translated_text'] = t
df.loc[df['question_id']==195, 'question'] = 'In the diagram'

In [21]:
t = """Image Annotation: The urban flow intensity in Guangdong is 615.2 billion yuan, and in Shanghai it is 227.4 billion yuan. Guizhou's urban flow is 15.6 billion yuan, and Tibet's is 500 million yuan.
The last five provinces/regions are Guizhou, Hainan, Ningxia, Qinghai, and Tibet.

Scenario: Urban flow refers to the movement of people, goods, information, capital, and technology between cities. The table below shows the urban flow intensity between some provinces/regions in China in 2008 (excluding Hong Kong, Macau, and Taiwan). Urban flow intensity is the impact of outward-oriented functions (agglomeration and radiation) of cities in regional urban interconnection. Read the table and answer the following questions.

Question: In the table...

Answers:

A) Urban flow intensity multiple: Guangdong/Shanghai > Guizhou/Tibet
B) The last five provinces/regions span three levels of my country's terrain.
C) Three provinces/regions have neither coastline nor land border.
D) Provinces/regions adjacent to Shandong include Jiangsu, Shanghai, and Beijing."""
df.loc[df['question_id']==976, 'translated_text'] = t
df.loc[df['question_id']==976, 'question'] = 'In the table...'

In [22]:
t = """Image Annotation: Sunlight shines directly on the ground, this is relationship (1)
The ground reflects light from the sun, this is relationship (3)
Clouds reflect light from the ground, this is relationship (4)
Clouds reflect light from the sun, this is relationship (2)
Sunlight and cloud-ground scattering diagram

Scenario: Global warming caused by the greenhouse effect has become a focus of human attention. Recently, Australian scientists pointed out that "global darkening" is also accompanied by global warming.

Question: Based on the diagram, answer: The link corresponding to "global darkening" is shown in the diagram.

Answers:

A) ①
B) ②
C) ③
D) ④"""
df.loc[df['question_id']==3037, 'translated_text'] = t
df.loc[df['question_id']==3037, 'question'] = 'Based on the diagram, answer: The link corresponding to "global darkening" is shown in the diagram.'

In [23]:
num_null_rows = df['question'].isnull().sum()

print(num_null_rows)

0


In [24]:
df.to_csv(path, index=False)

## Deal with missing annotations

In [25]:
num_null_rows = df['annotation'].isnull().sum()

print(num_null_rows)

null_rows = set(df[df["annotation"].isnull()]['scenario_id'])
print(len(null_rows))

43
21


In [27]:
with open(original_path, 'r') as f:
    json_data = json.load(f)
real_nanno = set()
# We look for the scenarios that are empty by default
for d in json_data:
    if d['free-form_annotation'] == "" and d['templated_annotation'] == "":
        real_nanno.add(d['scenario_id'])
        
## We check the scenarios that are missing because of the translation
print(f"Scenarios id where there is a problem : {null_rows.difference(real_nanno)}")

{447}


In [33]:
problem = df[df["annotation"].isnull()]
problem = problem[problem["scenario_id"] == 447]
print(len(problem))

zh_text = list(problem['text'])
en_text = list(problem['translated_text'])
ids = list(problem['question_id'])
for zh_t, en_t, idx in zip(zh_text, en_text, ids):
    print("\n---------------\n")
    print(idx)
    print(zh_t)
    print(en_t)

1

---------------

946
Image Annotation :
琵拉大沙丘

Scenario :
琵拉大沙丘坐落于法国西南部波尔多市(0°34′E ，44°50′N  )的大西洋畔，它的东边是郁郁葱葱的森林。琵拉沙丘以每年5米的速度持续向内陆推进，它吞没了部分房屋、道路乃至森林，给附近居民的生产生活造成诸多不便。据此完成下列各题。

Question :
琵拉大沙丘形成的主要原因是

Answers :
A) 盛行西风会将沙吹向岸边，使沙丘增生
B) 山地阻挡海洋水汽的深入
C) 副热带高压控制，盛行下沉气流，降水少 
D) 寒流流经具有降温减湿作用
Image Annotation:

Scenario:
The Dune of Pilat is located on the Atlantic coast near Bordeaux (0°34′E, 44°50′N) in southwestern France. Lush forests surround its eastern edge. The Dune of Pilat is advancing inland at a rate of 5 meters per year, engulfing houses, roads, and even forests, causing significant inconvenience to nearby residents. Answer the following questions based on this information.

Question:
What is the main reason for the formation of the Dune of Pilat?

Answers:
A) Prevailing westerly winds blow sand toward the shore, causing the dune to grow.
B) Mountains block the penetration of ocean moisture.
C) Subtropical high pressure dominates, resulting in prevailing downdrafts and low pr

In [34]:
t = """Image Annotation: Dune of Pilat

Scenario: The Dune of Pilat is located on the Atlantic coast of Bordeaux, southwestern France (0°34′E, 44°50′N), bordered by lush forests to the east. The Dune of Pilat continues to advance inland at a rate of 5 meters per year, engulfing some houses, roads, and even forests, causing considerable inconvenience to the lives and livelihoods of nearby residents. Answer the following questions based on this information.

Question: The main reason for the formation of the Dune of Pilat is:

Answers:

A) Prevailing westerly winds blow sand towards the shore, causing dune growth.
B) Mountains block the penetration of oceanic moisture.
C) The subtropical high pressure system controls the area, resulting in prevailing descending air currents and low rainfall.
D) Cold air masses have a cooling and dehumidifying effect."""
df.loc[df['question_id']==946, 'translated_text'] = t
df.loc[df['question_id']==946, 'annotation'] = 'Dune of Pilat'

In [36]:
print(len(df))
num_null_rows = df.isnull().any(axis=1).sum()

print(num_null_rows)
# There is 42 questions that don't have annotations by default corresponding to 21 scenarios
assert num_null_rows == 42

4110
42


In [37]:
df.to_csv(path, index=False)

## Check for scenario and choices

In [5]:
num_null_rows = df['scenario'].isnull().sum()

print(num_null_rows)

0


In [6]:
num_null_rows = df['A'].isnull().sum()

print(num_null_rows)

0


In [7]:
num_null_rows = df['B'].isnull().sum()

print(num_null_rows)

0


In [8]:
num_null_rows = df['C'].isnull().sum()

print(num_null_rows)

0


In [9]:
num_null_rows = df['D'].isnull().sum()

print(num_null_rows)

0


In [10]:
num_null_rows = df['annotation'].isnull().sum()

print(num_null_rows)

42


## Create a Dataset object

In [11]:
columns_to_keep = ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D']

def split_by_scenario(df, group_col="scenario_id",
                      test_size=0.20, val_size=0.15, random_state=42):
    # --- 1) Split train+val vs test ---
    gss = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=random_state)
    train_val_idx, test_idx = next(gss.split(df, groups=df[group_col]))

    df_train_val = df.iloc[train_val_idx]
    df_test = df.iloc[test_idx]

    # --- 2) Split train vs val (sur train_val uniquement) ---
    val_relative_size = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(test_size=val_relative_size, n_splits=1, random_state=random_state)
    train_idx, val_idx = next(gss2.split(df_train_val, groups=df_train_val[group_col]))

    df_train = df_train_val.iloc[train_idx]
    df_val = df_train_val.iloc[val_idx]

    return df_train, df_val, df_test

df = pd.read_csv(path)

# Exemple d'utilisation
df_train, df_val, df_test = split_by_scenario(df)

print("Train:", df_train["scenario_id"].nunique(), "scénarios")
print("Val:", df_val["scenario_id"].nunique(), "scénarios")
print("Test:", df_test["scenario_id"].nunique(), "scénarios")

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))


# Vérification : aucun scénario en commun
print("Overlap train/test:", set(df_train.scenario_id) & set(df_test.scenario_id))
print("Overlap val/test:", set(df_val.scenario_id) & set(df_test.scenario_id))
print("Overlap train/val:", set(df_train.scenario_id) & set(df_val.scenario_id))

dataset = datasets.DatasetDict({
        "train": datasets.Dataset.from_pandas(df_train[columns_to_keep]).remove_columns("__index_level_0__"),
        "validation": datasets.Dataset.from_pandas(df_val[columns_to_keep]).remove_columns("__index_level_0__"),
        "test": datasets.Dataset.from_pandas(df_test[columns_to_keep]).remove_columns("__index_level_0__")
    })
dataset.save_to_disk(os.path.join(output_path, "GeoSQA"))
print(dataset)
print(dataset['train'].features)

Train: 1287 scénarios
Val: 297 scénarios
Test: 397 scénarios
Train: 2644
Val: 628
Test: 838
Overlap train/test: set()
Overlap val/test: set()
Overlap train/val: set()


Saving the dataset (1/1 shards): 100%|██████████| 838/838 [00:00<00:00, 86067.55 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 2644
    })
    validation: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 628
    })
    test: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 838
    })
})
{'question_id': Value('int64'), 'scenario_id': Value('int64'), 'answer': Value('string'), 'annotation': Value('string'), 'scenario': Value('string'), 'question': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string')}
